# Expanded ACS missing-feature study: disjoint 2023 auxiliary data + MLP shadows

ACS 2023 is used for both the target and auxiliary source. The auxiliary pool
contains only unused records: all target train, validation, test, MIA member and
MIA nonmember IDs are excluded. Records may be reused across shadow models.

Retain Exp 3's 48 features, nominal PINCP > 50,000 label, target training size,
100-epoch target configuration, seven masks, K=6 and original blind walks.
The 16 shadows comprise four random forests, four gradient-boosting models,
four XGBoost models and four MLPs. Each has a separately seeded 40/20/40 split;
preprocessing fits its training records only. MLPs use the existing generic
shadow settings, not the target architecture or target training schedule.

Includes the complete-feature baseline, marginal PFV, and conditional
single/average/mean-only/uncertainty-aware comparisons. The historical shadow
versus target commuting-walk asymmetry is preserved and limits interpretation.
Same-year auxiliary data remove the year mismatch but provide a smaller pool;
this and the new algorithm mix mean results need not match the historical 62.625%.

No experiments have been run in this notebook. No checkpoints or logs are used.
Only final results, row IDs and protocols are saved in a separate artifact directory.


In [1]:
from pathlib import Path
import json
import time
import importlib
import src.expanded_data as expanded_data_module
import src.shadow_models as shadow_models_module
importlib.reload(expanded_data_module)
importlib.reload(shadow_models_module)
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.data import load_acs_income, split_acs_data, subset_target_training
from src.expanded_data import (
    load_expanded_acs, preprocess_expanded_partitions, expanded_feature_inventory,
    reference_split_config, EXPANDED_FEATURES, EXCLUDED, load_expanded_year,
)
from src.training import train_target_model, evaluate_model
from src.shadow_models import (load_adult_data, train_shadow_model, train_shadow_models,
                               summarize_shadow_models, PAPER_SHADOW_ALGORITHMS, MLP_SHADOW_CONFIG)
from src.same_year_mia import disjoint_same_year_pool
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import (collect_shadow_paper_features, collect_target_paper_features,
                                       make_probability_oracle)

torch.set_num_threads(4)
SEED = 42
TRAIN_SIZE = 100_000
INCOME_BASE_YEAR = None
NOMINAL_INCOME_METADATA = dict(enabled=False, label='PINCP > 50000', adjustment='None')
EXPERIMENT_DIR = Path('artifacts/exp_3_2023_2023_100000_nominal_mixed_mlp')
RUN_DIR = EXPERIMENT_DIR / 'reduced_k6_s16_m7'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED, 'Use the reference seed consistently.'
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0, seed=SEED,
    device='cpu', early_stopping=False, restore_best=False, monitor_train=True)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=SEED)
SHADOW_RECORDS_PER_CLASS = 2000
N_SHADOW_MODELS = 16
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS + ('mlp',)
SHADOW_MLP_CONFIG = dict(MLP_SHADOW_CONFIG)
assert N_SHADOW_MODELS == 4 * len(SHADOW_ALGORITHMS)
# Native commuting units; all other added codes are categorical in the walk.
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
print('Training size:', TRAIN_SIZE, '| Input fields:', len(EXPANDED_FEATURES))
print('Reference split configuration:', SPLIT_CONFIG)

import importlib
import src.notebook_output as notebook_output_module
importlib.reload(notebook_output_module)
notebook_output_module.suppress_sklearn_parallel_warning()
from src.notebook_output import CellProgress, quiet_output
assert len(EXPANDED_FEATURES) == 48 and 'VPS' in EXPANDED_FEATURES


Training size: 100000 | Input fields: 48
Reference split configuration: {'validation_size': 10000, 'test_size': 10000, 'mia_nonmember_size': 10000, 'mia_member_size': 10000, 'seed': 42}


## Load 2023 target records and make stratified partitions
Split sizes and seed follow Experiment 2, applied to the new 2023 records.


In [2]:
expanded_source = load_expanded_year(2023, income_base_year=INCOME_BASE_YEAR)
reference_partitions = split_acs_data(expanded_source, **SPLIT_CONFIG)
expanded_base_partitions = reference_partitions
expanded_partitions = subset_target_training(
    expanded_base_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(expanded_partitions[name].index, reference_partitions[name].index)
assert set(expanded_partitions['mia_members'].index).issubset(expanded_partitions['train'].index)
for name in ['validation', 'test', 'mia_nonmembers']:
    assert set(expanded_partitions[name].index).isdisjoint(expanded_partitions['train'].index)
print('Target income adjustment:', expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
print('Eligible records:', len(expanded_source))
print('Unused records:', len(expanded_source) - TRAIN_SIZE
      - sum(len(expanded_partitions[name]) for name in ['validation', 'test', 'mia_nonmembers']))
display(pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
    for name, part in expanded_partitions.items()}).T)
feature_inventory = expanded_feature_inventory(expanded_partitions['train'])
display(feature_inventory)


Target income adjustment: {'enabled': False, 'label': 'PINCP > 50000', 'adjustment': 'None'}
Eligible records: 203278
Unused records: 73278


,records,positive_rate
validation,10000.0,0.49840
test,10000.0,0.49840
mia_nonmembers,10000.0,0.49840
train,100000.0,0.49845
mia_members,10000.0,0.49840


,group,type,unique_nonmissing,missing_fraction
feature,,,,
AGEP,original_ACSIncome,numeric,75,0.00000
COW,original_ACSIncome,categorical,8,0.00000
SCHL,original_ACSIncome,categorical,24,0.00000
MAR,original_ACSIncome,categorical,5,0.00000
OCCP,original_ACSIncome,categorical,527,0.00000
POBP,original_ACSIncome,categorical,220,0.00000
WKHP,original_ACSIncome,numeric,97,0.00000
SEX,original_ACSIncome,categorical,2,0.00000
RAC1P,original_ACSIncome,categorical,9,0.00000


## Preprocess using training records only
The output is sparse; the MLP densifies only the current batch. Cardinality
counts above refer to the original fields, not one-hot dimensions.


In [3]:
expanded_preprocessor, expanded_arrays = preprocess_expanded_partitions(expanded_partitions)
print('Encoded input dimensions:', expanded_arrays['train']['X'].shape[1])
print('Training CSR storage (MiB):', round(sum(
    value.nbytes for value in [expanded_arrays['train']['X'].data,
    expanded_arrays['train']['X'].indices, expanded_arrays['train']['X'].indptr]) / 2**20, 2))
pd.DataFrame({name: {'rows': values['X'].shape[0], 'encoded_features': values['X'].shape[1]}
    for name, values in expanded_arrays.items()}).T


Encoded input dimensions: 2689
Training CSR storage (MiB): 38.53


,rows,encoded_features
validation,10000,2689
test,10000,2689
mia_nonmembers,10000,2689
train,100000,2689
mia_members,10000,2689


## Train the target
Fixed 100 epochs, final weights retained. Validation is monitored but does not
select an epoch in this deliberately overfit experimental condition.


In [4]:
target_started = time.monotonic()
with quiet_output():
    expanded_model, expanded_history, expanded_metadata = train_target_model(
        expanded_arrays['train']['X'], expanded_arrays['train']['y'],
        expanded_arrays['validation']['X'], expanded_arrays['validation']['y'], **TRAIN_CONFIG)
target_training_seconds = time.monotonic() - target_started
print('Target training seconds:', round(target_training_seconds, 1))
display(pd.DataFrame(expanded_history).iloc[::10])


Target training seconds: 933.2


,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss,train_eval_accuracy,train_eval_roc_auc,train_eval_log_loss
0,1,0.416451,0.8134,0.900563,0.403865,0.82923,0.913752,0.378947
10,11,0.060908,0.7923,0.867702,1.158196,0.98532,0.999131,0.039848
20,21,0.016433,0.7787,0.858233,1.512833,0.99233,0.999692,0.021763
30,31,0.011112,0.7940,0.868656,1.540580,0.99794,0.999976,0.006108
40,41,0.007754,0.7937,0.868837,1.735596,0.99766,0.999973,0.006455
50,51,0.007198,0.7898,0.869194,1.748980,0.99778,0.999985,0.005592
60,61,0.004103,0.7855,0.862731,1.791714,0.99801,0.999974,0.006090
70,71,0.005068,0.7877,0.863999,1.846895,0.99766,0.999985,0.006761
80,81,0.003399,0.7911,0.867223,1.839046,0.99867,0.999990,0.003930
90,91,0.003514,0.7906,0.865856,1.933827,0.99936,0.999998,0.001810


## Income-prediction evaluation
These are target income metrics, not membership metrics. The test set was not
used for model selection. Do not choose target settings using these results.


In [5]:
expanded_metrics = {name: evaluate_model(expanded_model, expanded_arrays[name]['X'],
    expanded_arrays[name]['y']) for name in ['train', 'validation', 'test']}
display(pd.DataFrame(expanded_metrics).T)
print('Train-test accuracy gap (percentage points):',
      100 * (expanded_metrics['train']['accuracy'] - expanded_metrics['test']['accuracy']))


,accuracy,roc_auc,log_loss
train,0.99849,0.999985,0.004646
validation,0.78650,0.861110,1.959403
test,0.79520,0.867629,1.864808


Train-test accuracy gap (percentage points): 20.328999999999997


## 16 shadows on unused ACS 2023 records
Exclude every target training and evaluation record before creating the pool.
Train four models per algorithm: random forest, gradient boosting, XGBoost and MLP.
Each shadow uses its own seed and randomized 40/20/40 train/validation/nonmember
split. Fit scaling, encoding and imputation strictly on that shadow's train split.
Models can reuse auxiliary records across shadows; grouped attack splits keep
repeated record IDs together. The complete-feature attack samples 2,000 IN and
2,000 OUT records per shadow, as in Exp 3.

Keep existing tree settings. MLP shadows use hidden layers (128, 64), dropout 0,
learning rate 0.001, weight decay 0.0001, batch size 256 and at most 50 epochs,
with validation-loss early stopping (patience 5) and best-weight restoration.
They share the target's preprocessing implementation and 48 input fields, but
are not an architecture-matched attack. The smaller disjoint auxiliary pool
also gives each shadow fewer training records than the target.


In [6]:
acs_auxiliary, auxiliary_disjointness_audit = disjoint_same_year_pool(
    expanded_source, expanded_partitions)
assert list(acs_auxiliary.columns) == list(expanded_source.columns)
assert set(EXPANDED_FEATURES) <= set(acs_auxiliary.columns)
print(f'Disjoint ACS 2023 auxiliary pool: {len(acs_auxiliary):,} records')
display(auxiliary_disjointness_audit)
print('Auxiliary income adjustment:', acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
assert auxiliary_disjointness_audit['overlapping_record_ids'].eq(0).all()
shadow_results = {}
shadow_progress = CellProgress()
for index in range(N_SHADOW_MODELS):
    algorithm = SHADOW_ALGORITHMS[index % len(SHADOW_ALGORITHMS)]
    name = f'{algorithm}_{index + 1:02d}'
    shadow_progress.update(f'Training shadow {index + 1}/{N_SHADOW_MODELS}: {name}')
    with quiet_output():
        result = train_shadow_model(acs_auxiliary, algorithm, seed=SEED + 10 * index, schema='expanded_acs',
            parameters=SHADOW_MLP_CONFIG if algorithm == 'mlp' else None)
    # Verify every actual shadow partition belongs to the audited pool.
    for partition_ids in result['partition_ids'].values():
        assert set(partition_ids).issubset(acs_auxiliary.index)
    result['attack_data']['shadow_model'] = name
    shadow_results[name] = result
shadow_progress.update(f'Completed {len(shadow_results)} shadows')
# Verify native-input queries reproduce probabilities computed during training.
for name, result in shadow_results.items():
    audit_rows = result['attack_data'].iloc[:32]
    native = acs_auxiliary.loc[audit_rows['record_id'], EXPANDED_FEATURES]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema='acs')
    np.testing.assert_allclose(query(native),
        audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(),
        rtol=1e-5, atol=1e-6, err_msg=f'Query mismatch for {name}')
print('Probability oracle consistency verified for all 16 shadows.')
display(summarize_shadow_models(shadow_results))


Disjoint ACS 2023 auxiliary pool: 73,278 records


,target_partition,target_records,shadow_pool_records,overlapping_record_ids
0,validation,10000,73278,0
1,test,10000,73278,0
2,mia_nonmembers,10000,73278,0
3,train,100000,73278,0
4,mia_members,10000,73278,0


Auxiliary income adjustment: {'enabled': False, 'label': 'PINCP > 50000', 'adjustment': 'None'}


Probability oracle consistency verified for all 16 shadows.


records  accuracy   roc_auc  log_loss
shadow_model         partition                                        
random_forest_01     train         29311  0.884071  0.959304  0.343611
                     validation    14656  0.804995  0.888235  0.438183
                     nonmembers    29311  0.799734  0.885836  0.440880
gradient_boosting_02 train         29311  0.819999  0.903942  0.397425
                     validation    14656  0.811681  0.892424  0.415422
                     nonmembers    29311  0.811402  0.892451  0.414991
xgboost_03           train         29311  0.837194  0.918542  0.367701
                     validation    14656  0.816935  0.897608  0.403711
                     nonmembers    29311  0.819453  0.900270  0.398949
mlp_04               train         29311  0.841698  0.918561  0.365011
                     validation    14656  0.806769  0.888316  0.421873
                     nonmembers    29311  0.812937  0.894948  0.409166
random_forest_05     train         29311  0.881410  0.956843  0.348735
                     validation    14656  0.798376  0.885129  0.442886
                     nonmembers    29311  0.803930  0.887333  0.439565
gradient_boosting_06 train         29311  0.818737  0.903509  0.398231
                     validation    14656  0.805882  0.888429  0.421383
                     nonmembers    29311  0.814711  0.894926  0.411248
xgboost_07           train         29311  0.835932  0.918139  0.368373
                     validation    14656  0.816799  0.899286  0.401389
                     nonmembers    29311  0.816826  0.898865  0.401965
mlp_08               train         29311  0.836102  0.914588  0.371843
                     validation    14656  0.810590  0.893481  0.413194
                     nonmembers    29311  0.813449  0.894621  0.410081
random_forest_09     train         29311  0.882877  0.957820  0.344822
                     validation    14656  0.804244  0.884870  0.442262
                     nonmembers    29311  0.797755  0.884108  0.442577
gradient_boosting_10 train         29311  0.821398  0.903646  0.397863
                     validation    14656  0.810931  0.893524  0.413647
                     nonmembers    29311  0.809457  0.892774  0.415041
xgboost_11           train         29311  0.836717  0.916754  0.371505
                     validation    14656  0.826215  0.905931  0.389988
                     nonmembers    29311  0.815871  0.899951  0.400332
mlp_12               train         29311  0.843949  0.920175  0.364151
                     validation    14656  0.814479  0.898409  0.403057
                     nonmembers    29311  0.816451  0.898178  0.403494
random_forest_13     train         29311  0.881649  0.958427  0.343120
                     validation    14656  0.803766  0.887932  0.437751
                     nonmembers    29311  0.800348  0.884564  0.441733
gradient_boosting_14 train         29311  0.817952  0.901002  0.401987
                     validation    14656  0.815025  0.896444  0.409314
                     nonmembers    29311  0.810208  0.892401  0.415838
xgboost_15           train         29311  0.837535  0.918056  0.368251
                     validation    14656  0.811272  0.897301  0.404046
                     nonmembers    29311  0.815223  0.899223  0.401098
mlp_16               train         29311  0.846235  0.924391  0.353684
                     validation    14656  0.809771  0.893306  0.413288
                     nonmembers    29311  0.814472  0.896154  0.407853

## Extract shadow features and fit the random forest attack
Features: class-scaled logit, blind walk, entropy and cross-entropy. Use 2,000
members and 2,000 non-members per shadow. Attack record-ID grouping prevents
repeated 2023 records from crossing attack partitions. Select both thresholds
on auxiliary validation only; no target membership labels train this classifier.


In [7]:
with quiet_output():
    shadow_attack_data, shadow_query_diagnostics = collect_shadow_paper_features(
        shadow_results, acs_auxiliary, per_class=SHADOW_RECORDS_PER_CLASS,
        config=WALK_CONFIG, seed=SEED)
display(pd.DataFrame(shadow_query_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper')
display(pd.DataFrame(attack_result['metrics']).T)


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
random_forest_01,119031,164000,0.53275,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_02,118785,164000,0.550375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_03,120375,164000,0.56,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_04,122070,164000,0.55925,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_05,121213,164000,0.543875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_06,118738,164000,0.5535,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_07,117627,164000,0.541375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_08,119700,164000,0.541875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_09,123170,164000,0.562125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_10,118074,164000,0.550125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,38483.0,19257.0,0.549437,0.549137,0.642166,0.037285,0.457388,3380.0,15846.0,1493.0,17764.0,0.592664,0.016565,0.003485
validation,12813.0,6495.0,0.514946,0.509784,0.508402,0.010316,0.457388,860.0,5458.0,757.0,5738.0,0.592664,0.010316,0.009180
test,12704.0,6248.0,0.506455,0.512502,0.526782,0.008643,0.457388,924.0,5532.0,738.0,5510.0,0.592664,0.007682,0.009294


## Final MIA evaluation on the expanded target
Query the expanded native input schema, preserving categorical codes and
handling numeric commute fields in native units. The walk still generates
field-valid but potentially implausible combinations, and its dimensionality
matches the expanded auxiliary schema. Thresholds are frozen from auxiliary validation.


In [8]:
with quiet_output():
    target_attack_data, target_query_diagnostics = collect_target_paper_features(
        expanded_model, expanded_preprocessor, expanded_partitions,
        config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
display(pd.Series(target_query_diagnostics, name='expanded target query diagnostics'))
target_mia_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
target_mia_metrics['tpr_at_accuracy_threshold'] = target_mia_metrics['tp'] / target_mia_metrics['members']
target_mia_metrics['fpr_at_accuracy_threshold'] = target_mia_metrics['fp'] / (
    target_mia_metrics['records'] - target_mia_metrics['members'])
display(pd.DataFrame({'2023 auxiliary attack test': attack_result['metrics']['test'],
                      'Expanded ACS target MIA': target_mia_metrics}).T)


record_queries                                                       526991
max_record_queries                                                   820000
walk_censored_fraction                                             0.438975
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: expanded target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold
2023 auxiliary attack test,12704.0,6248.0,0.506455,0.512502,0.526782,0.008643,0.457388,924.0,5532.0,738.0,5510.0,0.592664,0.007682,0.009294,NaN,NaN
Expanded ACS target MIA,20000.0,10000.0,0.562200,0.562200,0.566377,0.002700,0.457388,3302.0,6698.0,2058.0,7942.0,0.592664,0.001000,0.004100,0.7942,0.6698


## Small reproducibility artifacts
Save summaries, the feature inventory and row assignments. Models remain in memory;
no checkpoints or log files are loaded or saved.


In [9]:
result_dir = RUN_DIR
result_dir.mkdir(parents=True, exist_ok=True)
feature_inventory.to_csv(result_dir / 'feature_inventory.csv')
auxiliary_disjointness_audit.to_csv(result_dir / 'auxiliary_disjointness_audit.csv', index=False)
pd.Series(acs_auxiliary.index, name='record_id').to_csv(
    result_dir / 'auxiliary_pool_ids.csv', index=False)
for shadow_name, result in shadow_results.items():
    for partition_name, record_ids in result['partition_ids'].items():
        pd.Series(record_ids, name='record_id').to_csv(
            result_dir / f'{shadow_name}_{partition_name}_ids.csv', index=False)
for name, part in expanded_partitions.items():
    pd.Series(part.index, name='record_id').to_csv(result_dir / f'{name}_ids.csv', index=False)
summarize_shadow_models(shadow_results).to_csv(result_dir / 'shadow_metrics.csv')
summary = dict(income_base_year=INCOME_BASE_YEAR,
    target_income_adjustment=expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    auxiliary_income_adjustment=acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    perturbation_protocol_audit={'shadow_numeric_fields': {'AGEP': [1.0, 17, 99], 'WKHP': [2.0, 1, 99]}, 'target_extra_numeric_fields': {'JWMNP': [2.0, 0, 200], 'JWRIP': [1.0, 1, 10]}, 'shadow_commuting_fields': 'Categorical replacements, unchanged from original run', 'target_commuting_fields': 'Numeric perturbations, unchanged from original run', 'interpretation': 'Historical shadow/target commuting-field walk asymmetry is retained.'},
    walk_numeric_protocol_scope='Extra target numeric fields only',
    walk_numeric_protocol=EXPANDED_NUMERIC_WALK, execution_status='acs_baseline_completed',
    attack_metrics_status='oracle_consistency_verified', target_year=2023, auxiliary_year=2023, n_shadow_models=N_SHADOW_MODELS,
    shadow_algorithms=list(SHADOW_ALGORITHMS),
    shadow_mlp_config=SHADOW_MLP_CONFIG,
    shadow_split_fractions=dict(train=0.4, validation=0.2, nonmembers=0.4),
    shadow_records_per_membership_class=SHADOW_RECORDS_PER_CLASS,
    auxiliary_pool_records=len(acs_auxiliary),
    auxiliary_pool_source='unused ACS 2023 records after excluding all target partitions',
    auxiliary_disjointness_audit=auxiliary_disjointness_audit.to_dict('records'),
    shadow_seeds={name: result['seed'] for name, result in shadow_results.items()},
    original_feature_count=len(EXPANDED_FEATURES),
    encoded_feature_count=expanded_arrays['train']['X'].shape[1],
    train_size=TRAIN_SIZE, split_config=SPLIT_CONFIG, training=expanded_metadata,
    target_training_seconds=target_training_seconds, target_metrics=expanded_metrics,
    shadow_attack_metrics=attack_result['metrics'], target_mia_metrics=target_mia_metrics,
    target_query_diagnostics=target_query_diagnostics)
(result_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Summary and exact row IDs written to', result_dir)


Summary and exact row IDs written to artifacts\exp_3_2023_2023_100000_nominal_mixed_mlp\reduced_k6_s16_m7


## Missing-feature PFV baseline
Use seven preselected masks: two singles, two pairs, two triples and one quadruple.
These retain the reduced K=6 study; the complete-feature baseline uses the historical settings.
All 48 target inputs remain required; the target is not retrained after masking.
Each hidden field is replaced before querying, including before blind walks.
The reconstruction pool contains ACS 2023 predictors only, with no income-label
conditioning. Naturally missing auxiliary values are excluded from the sampling pool.
Both sources use the same year's feature codes. The original masks are retained;
related marriage and employment fields remain observed. Income labels remain known for the original attack's loss features.

The same complete-feature shadow attack is used in every condition, isolating the
effect of target completion. There is no mask-specific attack retraining. Auxiliary
attack train/validation/test grouping and thresholds therefore remain unchanged.
A single seed is a baseline realization, not a measure of reconstruction variability.


In [10]:
from itertools import combinations
from src.missing_feature_pfv import complete_pfv

MASK_FIELDS = ['MAR', 'WKHP', 'COW', 'SEX']
MISSING_MASKS = {
    'MAR': ['MAR'], 'WKHP': ['WKHP'],
    'MAR+COW': ['MAR', 'COW'], 'COW+SEX': ['COW', 'SEX'],
    'MAR+WKHP+SEX': ['MAR', 'WKHP', 'SEX'],
    'WKHP+COW+SEX': ['WKHP', 'COW', 'SEX'],
    'MAR+WKHP+COW+SEX': ['MAR', 'WKHP', 'COW', 'SEX'],
}
MISSING_DIR = RUN_DIR / 'missing_feature_pfv'
MISSING_DIR.mkdir(parents=True, exist_ok=True)
missing_rows = [{'condition': 'complete', 'n_missing': 0, **target_mia_metrics}]
missing_audits = {}
for mask_name, hidden in MISSING_MASKS.items():
    completed_partitions = {}
    # One shared completion pass: membership never selects a seed or distribution.
    candidates = pd.concat([expanded_partitions[name] for name in
        ['mia_members', 'mia_nonmembers']]).sort_index()
    completed_all = complete_pfv(candidates[EXPANDED_FEATURES],
        acs_auxiliary[EXPANDED_FEATURES], hidden, seed=SEED)
    completed_all['income_gt_50k'] = candidates['income_gt_50k']
    for name in ['mia_members', 'mia_nonmembers']:
        original = expanded_partitions[name]
        completed_partitions[name] = completed_all.loc[original.index].copy()
    with quiet_output():
        features, audit = collect_target_paper_features(
            expanded_model, expanded_preprocessor, completed_partitions,
            config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
    metrics, scores = evaluate_attack(attack_result['classifier'], features,
        threshold=attack_result['accuracy_threshold'],
        low_fpr_threshold=attack_result['low_fpr_threshold'])
    metrics['tpr_at_accuracy_threshold'] = metrics['tp'] / metrics['members']
    metrics['fpr_at_accuracy_threshold'] = metrics['fp'] / (metrics['records'] - metrics['members'])
    missing_rows.append({'condition': mask_name, 'n_missing': len(hidden), **metrics})
    missing_audits[mask_name] = audit
    features.assign(membership_score=scores).to_csv(MISSING_DIR / f'{mask_name}_scores.csv', index=False)
missing_comparison = pd.DataFrame(missing_rows).set_index('condition')
missing_comparison['auc_change_from_complete'] = missing_comparison['roc_auc'] - target_mia_metrics['roc_auc']
display(missing_comparison)
missing_comparison.to_csv(MISSING_DIR / 'comparison.csv')
(MISSING_DIR / 'protocol.json').write_text(json.dumps(dict(
    method='PFV_single_draw', auxiliary_year=2023, target_year=2023,
    masks=MISSING_MASKS, seed=SEED,
    naturally_missing_auxiliary_values='excluded',
    classifier='frozen complete-feature ACS 2023 attack',
    thresholds='frozen auxiliary validation', diagnostics=missing_audits,
    baseline_summary='../summary.json'), indent=2), encoding='utf-8')


,n_missing,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold,auc_change_from_complete
condition,,,,,,,,,,,,,,,,,,
complete,0,20000,10000,0.56220,0.56220,0.566377,0.0027,0.457388,3302,6698,2058,7942,0.592664,0.0010,0.0041,0.7942,0.6698,0.000000
MAR,1,20000,10000,0.56805,0.56805,0.577550,0.0037,0.457388,3431,6569,2070,7930,0.592664,0.0018,0.0039,0.7930,0.6569,0.011173
WKHP,1,20000,10000,0.55720,0.55720,0.563607,0.0076,0.457388,3675,6325,2531,7469,0.592664,0.0024,0.0041,0.7469,0.6325,-0.002771
MAR+COW,2,20000,10000,0.56605,0.56605,0.576428,0.0051,0.457388,3461,6539,2140,7860,0.592664,0.0021,0.0039,0.7860,0.6539,0.010051
COW+SEX,2,20000,10000,0.56540,0.56540,0.576667,0.0065,0.457388,3407,6593,2099,7901,0.592664,0.0015,0.0024,0.7901,0.6593,0.010290
MAR+WKHP+SEX,3,20000,10000,0.55635,0.55635,0.568061,0.0058,0.457388,3801,6199,2674,7326,0.592664,0.0030,0.0047,0.7326,0.6199,0.001684
WKHP+COW+SEX,3,20000,10000,0.55235,0.55235,0.562044,0.0102,0.457388,3727,6273,2680,7320,0.592664,0.0043,0.0048,0.7320,0.6273,-0.004333
MAR+WKHP+COW+SEX,4,20000,10000,0.55320,0.55320,0.560790,0.0071,0.457388,3818,6182,2754,7246,0.592664,0.0032,0.0036,0.7246,0.6182,-0.005587


3073

## Uncertainty-aware MF-MIA: study protocol
Research question: does reconstruction-induced score standard deviation improve
membership inference beyond the mean? The following cells are additional experiments;
they are not executed when this notebook is generated.

Use the seven preselected masks, target, 16 ACS 2023 shadows, four base
attack features and existing blind-walk protocol. The base attack is frozen.
Compare conditional single imputation, average score, learned mean-only RF and
matched mean+standard-deviation RF on identical completion draws. PFV above is a
separate reconstruction baseline, not the single-imputation control for this section.

Sequential forest-leaf donor sampling conditions each hidden field on visible
fields and earlier sampled hidden fields (MAR, WKHP, COW, SEX order). It preserves
observed auxiliary values but is an approximate predictive distribution, not a
calibrated posterior. Reconstruction training excludes all base-attack, meta-fit,
calibration and auxiliary-test candidate IDs. No income label conditions imputation.
Auxiliary validation IDs are divided into meta-fit and calibration groups. Auxiliary
test IDs remain untouched until evaluation. The base attack's earlier validation
thresholds are not reused for this comparison. No target labels select methods,
masks, K or thresholds. Income class labels remain known to the base loss feature.

K=6 and one reconstruction seed are predeclared exploratory settings, not a
confirmatory multi-seed study. All seven masks are reported without choosing a winner
on target results. Raw draws remain in memory during this run.
The original commuting walk asymmetry is retained and limits interpretation.
Use two single-field, two double-field, two triple-field and one four-field mask;
each of MAR, WKHP, COW and SEX is hidden in four masks. Six draws give noisier
variability estimates than the original K=20 setting. The base attack is refitted
using all 16 shadows, with grouped auxiliary splits recomputed from
that pool. Reconstruction/meta/calibration roles follow those new splits.
Results use a separate 2023-auxiliary mixed-MLP artifact root and reduced_k6_s16_m7 subdirectory.
Common walk seeds reduce additional Monte Carlo noise; score spread can still
include completion-dependent effects of the finite blind-walk procedure.

This section can be expensive: each candidate is evaluated K times per mask,
with up to 41 record queries per completion under the original walk settings.
DP training/evaluation and repeated-seed confirmation are not implemented here.

Execution uses memory only for reconstruction models, audit draws, completion
scores and aggregators. No checkpoints are loaded or saved. Run these cells in
order in one kernel session; restarting requires computing them again.
Only final metrics, row assignments and the protocol are written to disk.


In [11]:
# 1. Fixed configuration and disjoint auxiliary roles
import importlib
from sklearn.model_selection import train_test_split
from src.notebook_output import CellProgress, quiet_output
import src.uncertainty_mf_mia as uncertainty_module
# Pick up runtime-support edits without retraining/reloading the target or shadows.
importlib.reload(uncertainty_module)
from src.uncertainty_mf_mia import (
    ConditionalDonorChain, auxiliary_roles, completion_scores, summarize_scores,
    fit_aggregation, evaluate_aggregation,
)

UNCERTAINTY_K = 6
assert N_SHADOW_MODELS == len(shadow_results) == 16, 'Load the 16-shadow pool first.'
assert set(shadow_attack_data['shadow_model']) == set(shadow_results), 'Rebuild base attack data using all 16 shadows.'
assert set().union(*(set(part['shadow_model']) for part in attack_result['splits'].values())) == set(shadow_results), 'Refit the base attack on the 16-shadow pool.'
assert len(MISSING_MASKS) == 7, 'Use the seven preselected masks for the reduced study.'
RECONSTRUCTION_SEED = SEED + 1000
UNCERTAINTY_DIR = RUN_DIR / 'uncertainty_mf_mia'
UNCERTAINTY_DIR.mkdir(parents=True, exist_ok=True)
uncertainty_roles = auxiliary_roles(acs_auxiliary, attack_result, seed=SEED)
recon_fit_ids, recon_check_ids = train_test_split(
    sorted(uncertainty_roles['reconstruction']), test_size=0.2, random_state=SEED)
recon_training = acs_auxiliary.loc[recon_fit_ids, EXPANDED_FEATURES]
recon_check = acs_auxiliary.loc[recon_check_ids, EXPANDED_FEATURES]
display(pd.Series({name: len(ids) for name, ids in uncertainty_roles.items()}, name='unique_2023_records'))
uncertainty_reconstructors = {}


reconstruction    31306
base              25183
meta               4197
calibration        4197
test               8395
Name: unique_2023_records, dtype: int64

### Fit and audit conditional reconstruction using 2023 only
Held-out reconstruction checks use only reserved 2023 records, not attack candidates.
Numeric coverage is descriptive empirical coverage of the sampled 90% interval.
Categorical sample-match fraction measures draw accuracy; it is not calibrated
probability accuracy. Report these checks even when attack performance is poor.
Related fields remain visible, so reconstruction may be easy through proxies.


In [12]:
# 2. Fit one conditional reconstruction chain per mask; no target queries
from src.notebook_output import CellProgress, quiet_output
reconstruction_audit_rows = []
reconstruction_progress = CellProgress()
for mask_index, (mask_name, hidden) in enumerate(MISSING_MASKS.items(), start=1):
    mask_started = time.monotonic()
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: fitting reconstruction')
    with quiet_output():
        reconstructor = ConditionalDonorChain(hidden, seed=RECONSTRUCTION_SEED).fit(recon_training)
    uncertainty_reconstructors[mask_name] = reconstructor
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: auditing {UNCERTAINTY_K} completions')
    with quiet_output():
        samples = list(reconstructor.iter_samples(recon_check,
            seeds=[RECONSTRUCTION_SEED + draw for draw in range(UNCERTAINTY_K)]))
    for field in hidden:
        observed = recon_check[field].notna().to_numpy()
        if not observed.any():
            continue
        draws = np.column_stack([sample[field].to_numpy()[observed] for sample in samples])
        truth = recon_check[field].to_numpy()[observed]
        row = dict(mask=mask_name, field=field, records=int(observed.sum()))
        if field == 'WKHP':
            lower, upper = np.quantile(draws, [0.05, 0.95], axis=1)
            row.update(mae_of_sample_mean=float(np.abs(draws.mean(axis=1) - truth).mean()),
                sampled_90pct_coverage=float(((truth >= lower) & (truth <= upper)).mean()),
                mean_interval_width=float((upper - lower).mean()))
        else:
            row['sample_match_fraction'] = float((draws == truth[:, None]).mean())
        reconstruction_audit_rows.append(row)
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: complete ({time.monotonic() - mask_started:.1f}s)')
reconstruction_audit = pd.DataFrame(reconstruction_audit_rows)
display(reconstruction_audit.round(4))
reconstruction_audit.to_csv(UNCERTAINTY_DIR / 'reconstruction_audit.csv', index=False)


,mask,field,records,sample_match_fraction,mae_of_sample_mean,sampled_90pct_coverage,mean_interval_width
0,MAR,MAR,6262,0.6177,NaN,NaN,NaN
1,WKHP,WKHP,6262,NaN,8.7871,0.7359,22.4737
2,MAR+COW,MAR,6262,0.6273,NaN,NaN,NaN
3,MAR+COW,COW,6262,0.4769,NaN,NaN,NaN
4,COW+SEX,COW,6262,0.4749,NaN,NaN,NaN
5,COW+SEX,SEX,6262,0.5344,NaN,NaN,NaN
6,MAR+WKHP+SEX,MAR,6262,0.6335,NaN,NaN,NaN
7,MAR+WKHP+SEX,WKHP,6262,NaN,8.8144,0.7400,22.9034
8,MAR+WKHP+SEX,SEX,6262,0.5308,NaN,NaN,NaN
9,WKHP+COW+SEX,WKHP,6262,NaN,8.9195,0.7365,22.8918


### Generate held-out auxiliary completion scores
The original 2,000 member/nonmember samples per shadow are reused. Only meta-fit,
calibration and auxiliary-test records are queried here; base-attack training
records are excluded. Repeated IDs retain the same role across all shadows.
Completion never receives membership labels as a conditioning variable.


In [13]:
# 3. K scores per held-out shadow observation, separately for every mask
from src.notebook_output import CellProgress, quiet_output
uncertainty_auxiliary_tables = {}
uncertainty_query_audits = {}
auxiliary_progress = CellProgress()
heldout_ids = set().union(*(uncertainty_roles[role] for role in ['meta', 'calibration', 'test']))
for mask_name, reconstructor in uncertainty_reconstructors.items():
    mask_tables = []
    for shadow_name, shadow in shadow_results.items():
        selected = shadow_attack_data.loc[
            (shadow_attack_data['shadow_model'] == shadow_name)
            & shadow_attack_data['record_id'].isin(heldout_ids),
            ['record_id', 'membership']].sort_values('record_id').reset_index(drop=True)
        if selected.empty:
            continue
        assert selected['record_id'].is_unique
        candidates = acs_auxiliary.loc[selected['record_id']]
        oracle = make_probability_oracle(shadow['model'], shadow['preprocessor'], schema='acs')
        auxiliary_progress.update(f'Auxiliary scoring: {mask_name} / {shadow_name}, {len(selected):,} records, K={UNCERTAINTY_K}')
        with quiet_output():
            draw_scores, audit = completion_scores(oracle, candidates, reconstructor,
                attack_result['classifier'], k=UNCERTAINTY_K, seed=RECONSTRUCTION_SEED,
                walk_config=WALK_CONFIG)  # preserve existing shadow walk protocol
        table = summarize_scores(draw_scores)
        table['record_id'] = selected['record_id'].to_numpy()
        table['membership'] = selected['membership'].to_numpy()
        table['shadow_model'] = shadow_name
        table['role'] = [next(role for role in ['meta', 'calibration', 'test']
            if record_id in uncertainty_roles[role]) for record_id in table['record_id']]
        mask_tables.append(table)
        uncertainty_query_audits[f'{mask_name}/{shadow_name}'] = audit
    uncertainty_auxiliary_tables[mask_name] = pd.concat(mask_tables, ignore_index=True)
auxiliary_progress.update(f'Auxiliary scoring complete: {len(uncertainty_auxiliary_tables)} masks')


In [14]:
# 4. Fit aggregators, freeze thresholds on calibration, evaluate auxiliary test
from src.notebook_output import quiet_output
uncertainty_aggregators = {}
uncertainty_auxiliary_result_rows = []
for mask_name, table in uncertainty_auxiliary_tables.items():
    meta = table.loc[table['role'] == 'meta']
    calibration = table.loc[table['role'] == 'calibration']
    auxiliary_test = table.loc[table['role'] == 'test']
    with quiet_output():
        learned, thresholds = fit_aggregation(meta, calibration, seed=SEED)
    uncertainty_aggregators[mask_name] = (learned, thresholds)
    with quiet_output():
        metrics, predictions = evaluate_aggregation(auxiliary_test, learned, thresholds)
    uncertainty_auxiliary_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2023_auxiliary_test').to_dict('records'))
print('Auxiliary test: macro averages across masks of the same size')
display(pd.DataFrame(uncertainty_auxiliary_result_rows).groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))


Auxiliary test: macro averages across masks of the same size


roc_auc  balanced_accuracy  tpr_at_1pct_fpr_roc
n_missing method                                                       
1         average        0.5153             0.5133               0.0096
          learned_mean   0.5086             0.5078               0.0098
          single         0.5119             0.5097               0.0081
          uncertainty    0.5103             0.5022               0.0106
2         average        0.5138             0.5104               0.0105
          learned_mean   0.4985             0.4988               0.0106
          single         0.5148             0.5075               0.0091
          uncertainty    0.4970             0.4995               0.0092
3         average        0.5141             0.5097               0.0110
          learned_mean   0.5009             0.5016               0.0085
          single         0.5102             0.5091               0.0101
          uncertainty    0.5009             0.4986               0.0090
4         average        0.5083             0.5057               0.0082
          learned_mean   0.5009             0.5023               0.0072
          single         0.5019             0.5027               0.0069
          uncertainty    0.5125             0.5140               0.0093

### Final target evaluation: frozen methods and thresholds
Use the same target member/nonmember IDs as the complete-feature baseline.
Membership labels are attached only after scores are generated. The same K draws
feed single, averaging and both learned aggregators. Do not tune on these results.
The complete-feature and PFV results above are context; the controlled test for
the standard-deviation signal compares uncertainty against learned_mean here.


In [15]:
# 5. Query the existing ACS 2023 target; no target retraining
from src.notebook_output import CellProgress, quiet_output
uncertainty_result_rows = list(uncertainty_auxiliary_result_rows)
target_candidates = pd.concat([expanded_partitions[name]
    for name in ['mia_members', 'mia_nonmembers']]).sort_index()
target_member_ids = set(expanded_partitions['mia_members'].index)
target_oracle = make_probability_oracle(expanded_model, expanded_preprocessor, schema='acs')
target_progress = CellProgress()
for mask_name, reconstructor in uncertainty_reconstructors.items():
    target_progress.update(f'Target scoring: {mask_name}, K={UNCERTAINTY_K}')
    with quiet_output():
        draw_scores, audit = completion_scores(target_oracle, target_candidates,
            reconstructor, attack_result['classifier'], k=UNCERTAINTY_K,
            seed=RECONSTRUCTION_SEED, walk_config=WALK_CONFIG,
            numeric_protocol=EXPANDED_NUMERIC_WALK)
    table = summarize_scores(draw_scores)
    table['record_id'] = target_candidates.index.to_numpy()
    table['membership'] = table['record_id'].isin(target_member_ids).astype(int)
    learned, thresholds = uncertainty_aggregators[mask_name]
    with quiet_output():
        metrics, predictions = evaluate_aggregation(table, learned, thresholds)
    uncertainty_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2023_target').to_dict('records'))
    uncertainty_query_audits[f'{mask_name}/target'] = audit
uncertainty_results = pd.DataFrame(uncertainty_result_rows)
uncertainty_results.to_csv(UNCERTAINTY_DIR / 'comparison.csv', index=False)
target_progress.update(f'Target scoring complete: {len(uncertainty_reconstructors)} masks')
target_summary = uncertainty_results.loc[uncertainty_results['dataset'] == '2023_target']
print('Target: macro averages across masks of the same size')
display(target_summary.groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))
print('Target ROC-AUC by mask; complete metrics are saved in comparison.csv')
display(target_summary.pivot(index='mask', columns='method', values='roc_auc').round(4))
_ = (UNCERTAINTY_DIR / 'protocol.json').write_text(json.dumps(dict(
    k=UNCERTAINTY_K, reconstruction_seed=RECONSTRUCTION_SEED, masks=MISSING_MASKS,
    n_shadow_models=N_SHADOW_MODELS, shadow_names=list(shadow_results),
    mask_selection='preselected coverage: 2 singles, 2 pairs, 2 triples, 1 quadruple; each field hidden four times',
    reconstruction='sequential random-forest leaf donor sampling; approximate predictive distribution',
    reconstruction_chain_order=MASK_FIELDS, score_std_ddof=0,
    auxiliary_year=2023, target_year=2023, target_labels_for_tuning=False,
    meta_source='half of base-attack validation record groups',
    calibration_source='other half of base-attack validation record groups',
    methods=['single', 'average', 'learned_mean', 'uncertainty'],
    queries=uncertainty_query_audits,
    limitations=['one reconstruction seed', 'no DP evaluation yet',
        'smaller disjoint same-year auxiliary pool; shadows train on fewer records than target', 'original blind-walk asymmetry retained',
        'no confirmatory significance claim; repeated shadow rows require grouped inference']),
    indent=2), encoding='utf-8')


Target: macro averages across masks of the same size


roc_auc  balanced_accuracy  tpr_at_1pct_fpr_roc
n_missing method                                                       
1         average        0.5709             0.5076               0.0044
          learned_mean   0.4968             0.5092               0.0046
          single         0.5639             0.5029               0.0055
          uncertainty    0.4720             0.4956               0.0062
2         average        0.5820             0.5127               0.0056
          learned_mean   0.4943             0.5032               0.0047
          single         0.5664             0.4839               0.0065
          uncertainty    0.5050             0.5154               0.0119
3         average        0.5865             0.5015               0.0090
          learned_mean   0.5248             0.5182               0.0115
          single         0.5614             0.5496               0.0096
          uncertainty    0.5076             0.4960               0.0085
4         average        0.5870             0.5670               0.0105
          learned_mean   0.5219             0.5192               0.0097
          single         0.5602             0.5489               0.0108
          uncertainty    0.4934             0.4994               0.0070

Target ROC-AUC by mask; complete metrics are saved in comparison.csv


method,average,learned_mean,single,uncertainty
mask,,,,
COW+SEX,0.5821,0.5130,0.5648,0.5196
MAR,0.5715,0.4748,0.5658,0.4347
MAR+COW,0.5820,0.4756,0.5679,0.4903
MAR+WKHP+COW+SEX,0.5870,0.5219,0.5602,0.4934
MAR+WKHP+SEX,0.5883,0.5193,0.5630,0.4827
WKHP,0.5703,0.5188,0.5620,0.5094
WKHP+COW+SEX,0.5848,0.5303,0.5598,0.5326
